In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
dimensions = [1000, 500, 200, 100, 50, 20, 10, 5]
seeds = [0]

In [ ]:
static_kan_A, dyn_kan_A, mlp_A = torch.load('static_kan_A.pt', weights_only=False), torch.load('dyn_A.pt', weights_only=False), torch.load('mlp_A.pt', weights_only=False)

In [ ]:
def plot_r2s_rmses(static_kan, dyn_kan, mlp):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 5))

    def mean_err(results_all_seeds, dimensions, seeds, key):
        means = []
        lower_errs = []
        upper_errs = []
        
        for d in dimensions:
            final_r2s = np.array([results_all_seeds[seed][d][key][-1] for seed in seeds])
            
            mean_val = np.mean(final_r2s)
            min_val = np.min(final_r2s)
            max_val = np.max(final_r2s)
            
            means.append(mean_val)
            lower_errs.append(mean_val - min_val)
            upper_errs.append(max_val - mean_val)
            
        return np.array(means), np.vstack([lower_errs, upper_errs])

    static_means_r2, static_yerr_r2 = mean_err(static_kan, dimensions, seeds, 'r2s')
    dyn_means_r2, dyn_yerr_r2 = mean_err(dyn_kan, dimensions, seeds, 'r2s')
    mlp_means_r2, mlp_yerr_r2 = mean_err(mlp, dimensions, seeds, 'r2s')
    static_means_rmse, static_yerr_rmse = mean_err(static_kan, dimensions, seeds, 'rmses')
    dyn_means_rmse, dyn_yerr_rmse = mean_err(dyn_kan, dimensions, seeds, 'rmses')
    mlp_means_rmse, mlp_yerr_rmse = mean_err(mlp, dimensions, seeds, 'test_loss')

    ax1.errorbar(dimensions, static_means_r2, yerr=static_yerr_r2, marker='s', linestyle='-', 
                capsize=4, elinewidth=1.5, alpha=0.8, label='Static KAN G=5')  
    ax1.errorbar(dimensions, dyn_means_r2, yerr=dyn_yerr_r2, marker='^', linestyle='-', 
                capsize=4, elinewidth=1.5, alpha=0.8, label='Dynamic KAN G=[3, 5, 10, 20, 50, 100]')    
    ax1.errorbar(dimensions, mlp_means_r2, yerr=mlp_yerr_r2, marker='o', linestyle='-', 
                capsize=4, elinewidth=1.5, alpha=0.8, label='MLP')

    cutoff=3
    ax2.errorbar(dimensions[cutoff:], static_means_r2[cutoff:], yerr=static_yerr_r2[:, cutoff:], marker='s', linestyle='-',
                capsize=4, elinewidth=1.5, alpha=0.8)
    ax2.errorbar(dimensions[cutoff:], dyn_means_r2[cutoff:], yerr=dyn_yerr_r2[:, cutoff:], marker='^', linestyle='-',
                capsize=4, elinewidth=1.5, alpha=0.8)
    # ax2.errorbar(dimensions[cutoff:], mlp_means_r2[cutoff:], yerr=mlp_yerr_r2[:, cutoff:], marker='^', linestyle='-',
                # capsize=4, elinewidth=1.5, alpha=0.8)

    ax3.errorbar(dimensions, static_means_rmse, yerr=static_yerr_rmse, marker='s', linestyle='-',
                capsize=4, elinewidth=1.5, alpha=0.8, )   
    ax3.errorbar(dimensions, dyn_means_rmse, yerr=dyn_yerr_rmse, marker='^', linestyle='-',
                capsize=4, elinewidth=1.5, alpha=0.8)  
    ax3.errorbar(dimensions, mlp_means_rmse, yerr=mlp_yerr_rmse, marker='o', linestyle='-', label='MLP',
                capsize=4, elinewidth=1.5, alpha=0.8)

    ax1.set_xlabel('dimensions (log)')
    ax1.set_xscale('log')
    ax1.set_yscale('log')
    ax1.set_ylabel('R2 Score')
    ax1.axvline(x=100, color='red', linestyle='--', alpha=0.7)
    ax1.set_xticks(dimensions)
    ax1.set_xticklabels(dimensions)

    ax2.set_xlabel('dimensions (log)')
    ax2.set_xscale('log')
    ax2.set_yscale('log')
    ax2.set_ylabel('R2 Score')
    ax2.set_xticks(dimensions[cutoff:])
    ax2.set_xticklabels(dimensions[cutoff:])
    # ax2.get_yaxis().get_major_formatter().set_useOffset(False)

    ax3.set_xlabel('dimensions (log)')
    ax3.set_xscale('log')
    ax3.set_yscale('log')
    ax3.set_ylabel('RMSE (log)')

    handles, labels = ax1.get_legend_handles_labels()

    fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, -0.05), ncol=3)
    plt.suptitle('KAN vs MLP Performance with Increasing Dimensionality (Function A)')
    plt.tight_layout()
    plt.show()

In [ ]:
plot_r2s_rmses(static_kan_A, dyn_kan_A, mlp_A)